# 22. 라이브러리 규칙의 실제 약물 데이터베이스 빈도 확인

## 이번 노트북에서 할 것
- ChEMBL API로 승인된 약물(max_phase=4) SMILES 세트 확보
- 우리 라이브러리 14개 규칙 각각이 이 "실제 약물유사 화합물" 집합에서
  얼마나 자주 나타나는지 확인
- Tox21(산업화학물질 포함, 일반 화학물질 셋)에서의 빈도와 비교
- 학생 질문에 답: "우리가 다루는 toxicophore가 실제 신약개발 맥락에서도
  의미 있는 빈도로 나타나는가, 아니면 Tox21 특유의 비-약물성 화학물질에서만
  흔한 것인가"

## 간략한 정리 (21까지)
- 라이브러리 14개 규칙 확정, 전수 회귀 테스트 통과 (7가지 편집 방식)
- 신규: het-C-het_not_in_ring(오르토에스터->에스터, remove_substituent 방식)
- 학생 질문 제기: Tox21은 농약/산업화학물질을 포함하므로, 우리가 다루는
  독성 작용기가 "실제 신약 후보"에도 흔한지 확인 필요 - 안 그러면 실용성이
  낮을 수 있음
- test set은 여전히 미사용, valid set(seed=7)으로 개발/검증 중

## 다음에 해야 할 것 (오늘 끝나면)
- 빈도 확인 결과에 따라, 실용성 낮은 규칙은 "한계"로 명시하거나 우선순위
  조정 고려
- 최종 valid set 재검증 (14개 규칙 전부 반영)
- 제안서 반영, 문헌형 확장은 학생 진행 병행

In [1]:
# 셀 1
!pip install rdkit -q
!pip install chembl_webresource_client -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00


In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 245, done.
remote: Counting objects: 100% (245/245), done.
remote: Compressing objects: 100% (170/170), done.
remote: Total 245 (delta 129), reused 172 (delta 69), pack-reused 0 (from 0)
Receiving objects: 100% (245/245), 621.55 KiB | 13.22 MiB/s, done.
Resolving deltas: 100% (129/129), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
import importlib
from rdkit import Chem

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector
import src.tools.agent

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix

data = load_tox21_clean(random_state=7)
print("도구 로드 확인 완료")

[07:57:05] WARNING: not removing hydrogen atom without neighbors
[07:57:05] Explicit valence for atom # 8 Al, 6, is greater than permitted
[07:57:05] Explicit valence for atom # 3 Al, 6, is greater than permitted
[07:57:05] Explicit valence for atom # 4 Al, 6, is greater than permitted
[07:57:05] Explicit valence for atom # 4 Al, 6, is greater than permitted
[07:57:05] Explicit valence for atom # 9 Al, 6, is greater than permitted
[07:57:05] Explicit valence for atom # 5 Al, 6, is greater than permitted
[07:57:06] Explicit valence for atom # 16 Al, 6, is greater than permitted
[07:57:06] Explicit valence for atom # 20 Al, 6, is greater than permitted
[07:57:06] WARNING: not removing hydrogen atom without neighbors


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개
도구 로드 확인 완료


In [5]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule

def fetch_chembl_smiles(max_phase_value, limit=500):
    """지정한 max_phase(0~4)에 해당하는 화합물의 canonical SMILES를 가져옴."""
    results = molecule.filter(max_phase=max_phase_value).only(
        ['molecule_structures']
    )[:limit]
    smiles_list = []
    for r in results:
        struct = r.get('molecule_structures')
        if struct and struct.get('canonical_smiles'):
            smiles_list.append(struct['canonical_smiles'])
    return smiles_list

# 0: 전임상/hit 단계 후보물질, 4: 승인된 약물
print("전임상 단계(phase 0) 화합물 가져오는 중...")
smiles_phase0 = fetch_chembl_smiles(0, limit=500)
print(f"  {len(smiles_phase0)}개 확보")

print("승인 약물(phase 4) 가져오는 중...")
smiles_phase4 = fetch_chembl_smiles(4, limit=500)
print(f"  {len(smiles_phase4)}개 확보")

전임상 단계(phase 0) 화합물 가져오는 중...
  0개 확보
승인 약물(phase 4) 가져오는 중...
  498개 확보


In [6]:
# max_phase 값 분포 확인
sample_check = molecule.filter().only(['max_phase'])[:20]
for r in sample_check:
    print(r.get('max_phase'))

None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None


In [8]:
# 더 가벼운 방식: 필드 이름만 빠르게 확인
r = molecule.filter()[0]
print(list(r.keys()))

['atc_classifications', 'availability_type', 'biotherapeutic', 'black_box_warning', 'chemical_probe', 'chirality', 'cross_references', 'dosed_ingredient', 'first_approval', 'first_in_class', 'helm_notation', 'inorganic_flag', 'max_phase', 'molecule_chembl_id', 'molecule_hierarchy', 'molecule_properties', 'molecule_structures', 'molecule_synonyms', 'molecule_type', 'natural_product', 'oral', 'orphan', 'parenteral', 'polymer_flag', 'pref_name', 'prodrug', 'structure_type', 'therapeutic_flag', 'topical', 'usan_stem', 'usan_stem_definition', 'usan_substem', 'usan_year', 'veterinary', 'withdrawn_flag']


In [9]:
r = molecule.filter()[0]
print("max_phase:", r.get('max_phase'))
print("pref_name:", r.get('pref_name'))

max_phase: None
pref_name: None


In [10]:
all_rules_check = list(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'].keys())

def count_rule_frequency(smiles_list, rules):
    counts = {r: 0 for r in rules}
    valid_count = 0
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        if mol is None:
            continue
        valid_count += 1
        problems = detect_toxicophores(s)
        found = set(p['rule_name'] for p in problems)
        for r in rules:
            if r in found:
                counts[r] += 1
    return counts, valid_count

print("승인 약물(phase 4, 498개) 분석 중...")
counts_drug, n_drug = count_rule_frequency(smiles_phase4, all_rules_check)

print("Tox21(일반 화학물질, 표본 500개) 분석 중...")
counts_tox21, n_tox21 = count_rule_frequency(list(data['smiles_valid'][:500]), all_rules_check)

print(f"\n{'규칙':25s} | {'승인약물 (n='+str(n_drug)+')':>18s} | {'Tox21 (n='+str(n_tox21)+')':>15s}")
for rule in all_rules_check:
    drug_pct = counts_drug[rule] / n_drug * 100
    tox21_pct = counts_tox21[rule] / n_tox21 * 100
    print(f"{rule:25s} | {counts_drug[rule]:3d}개 ({drug_pct:5.1f}%)   | {counts_tox21[rule]:3d}개 ({tox21_pct:5.1f}%)")

승인 약물(phase 4, 498개) 분석 중...
Tox21(일반 화학물질, 표본 500개) 분석 중...

규칙                        |       승인약물 (n=498) |   Tox21 (n=500)
nitro_group               |  11개 (  2.2%)   |  30개 (  6.0%)
aldehyde                  |   1개 (  0.2%)   |  10개 (  2.0%)
Michael_acceptor_1        |   9개 (  1.8%)   |  22개 (  4.4%)
acid_halide               |   0개 (  0.0%)   |   3개 (  0.6%)
alkyl_halide              |   9개 (  1.8%)   |  24개 (  4.8%)
aniline                   |  19개 (  3.8%)   |  19개 (  3.8%)
Sulfonic_acid_2           |   2개 (  0.4%)   |  13개 (  2.6%)
imine_1_oxime             |   1개 (  0.2%)   |   1개 (  0.2%)
imine_1_general           |  13개 (  2.6%)   |  15개 (  3.0%)
catechol                  |   7개 (  1.4%)   |   6개 (  1.2%)
Thiocarbonyl_group        |   5개 (  1.0%)   |   6개 (  1.2%)
thiol_2                   |   2개 (  0.4%)   |   3개 (  0.6%)
thiol_1                   |   0개 (  0.0%)   |   2개 (  0.4%)
het-C-het_not_in_ring     |   2개 (  0.4%)   |   4개 (  0.8%)


In [11]:
uncovered_counts = {}
for s in smiles_phase4:
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        continue
    problems = detect_toxicophores(s)
    for p in problems:
        rule = p['rule_name']
        if get_replacement_candidates(rule) is None:  # 우리 라이브러리에 없는 규칙
            uncovered_counts[rule] = uncovered_counts.get(rule, 0) + 1

sorted_uncovered = sorted(uncovered_counts.items(), key=lambda x: -x[1])
print("승인 약물에서 발견되지만 우리 라이브러리에 없는 규칙 (빈도순 상위 15개):")
for rule, count in sorted_uncovered[:15]:
    pct = count / len(smiles_phase4) * 100
    print(f"  {rule}: {count}개 ({pct:.1f}%)")

승인 약물에서 발견되지만 우리 라이브러리에 없는 규칙 (빈도순 상위 15개):
  Aliphatic_long_chain: 46개 (9.2%)
  Oxygen-nitrogen_single_bond: 25개 (5.0%)
  isolated_alkene: 24개 (4.8%)
  beta-keto/anhydride: 20개 (4.0%)
  stilbene: 9개 (1.8%)
  imine_2: 9개 (1.8%)
  phosphor: 8개 (1.6%)
  quaternary_nitrogen_2: 7개 (1.4%)
  catechol_A(92): 7개 (1.4%)
  triple_bond: 7개 (1.4%)
  phthalimide: 7개 (1.4%)
  hydroquinone: 6개 (1.2%)
  hydroxamic_acid: 5개 (1.0%)
  quinone_A(370): 4개 (0.8%)
  azo_A(324): 4개 (0.8%)


In [ ]:
# 이후에 여기부터 추가 필요. PubChem등을 이용하여 추가하면 좋다!